In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

In [2]:
fake = Faker()

# Set random seed for generating same data each time 
Faker.seed(42)
np.random.seed(42)
random.seed(42)

NUM_PRODUCTIONS = 15000
NUM_SALES = 7500  # allow multiple sales per production

In [3]:

# Sample values for realism
crop_types = ['Wheat', 'Corn', 'Barley', 'Soybean', 'Potato']
varieties = {'Wheat': ['Durum', 'Emmer'], 'Corn': ['Sweet', 'Dent'], 'Barley': ['Hulled', 'Hulless'],
             'Soybean': ['Yellow', 'Black'], 'Potato': ['Russet', 'Red']}

fertilizers = ['NPK 15-15-15', 'Urea', 'Compost']
pesticides = ['Glyphosate', 'Chlorpyrifos', 'Neem Oil']
irrigation_types = ['Drip', 'Sprinkler', 'Flood']

buyer_types = ['Retailer', 'Wholesaler', 'Exporter']
channel_types = ['Direct', 'Online', 'Cooperative']

regions = ['Bavaria', 'Brandenburg', 'Saxony', 'Hesse', 'Lower Saxony']


In [ ]:
def generate_production_data():
    records = []
    for i in range(1, NUM_PRODUCTIONS + 1):
        crop = random.choice(crop_types)
        variety = random.choice(varieties[crop])
        planting_date = fake.date_between(start_date='-2y', end_date='-6m')
        
        #  harvest_date is not in the future
        harvest_raw = planting_date + timedelta(days=random.randint(90, 160))
        harvest_date = min(harvest_raw, datetime.today().date())
        
        yield_kg = random.uniform(1000, 10000)
        avg_yield = yield_kg / random.uniform(0.5, 2.5)
        expected_price = round(random.uniform(0.3, 1.5), 2)
        
        records.append({
            'production_id': i,
            'farm_name': fake.company(),
            'field_id': f'F-{random.randint(100, 999)}',
            'field_location': fake.city(),
            'crop_type': crop,
            'crop_variety': variety,
            'planting_date': planting_date,
            'harvest_date': harvest_date,
            'fertilizer_type': random.choice(fertilizers),
            'fertilizer_quantity_kg': round(random.uniform(50, 300), 1),
            'pesticide_type': random.choice(pesticides),
            'pesticide_quantity_ltr': round(random.uniform(5, 30), 1),
            'irrigation_type': random.choice(irrigation_types),
            'irrigation_quantity_ltr': round(random.uniform(500, 2000), 1),
            'total_yield_kg': round(yield_kg, 1),
            'avg_yield_per_hectare': round(avg_yield, 1),
            'soil_ph': round(random.uniform(5.5, 7.5), 2),
            'rainfall_mm': round(random.uniform(200, 800), 1),
            'avg_temperature_c': round(random.uniform(12, 26), 1),
            'labor_hours': round(random.uniform(50, 200), 1),
            'number_of_workers': random.randint(2, 10),
            'machinery_used': random.choice(['Tractor', 'Harvester', 'Plough']),
            'expected_market_price_per_kg': expected_price,
            'expected_total_revenue_eur': round(expected_price * yield_kg, 2)
        })
    return pd.DataFrame(records)

In [34]:

def generate_sales_data(production_df):
    records = []
    for i in range(1, NUM_SALES + 1):
        prod = production_df.sample(1).iloc[0]
        
        harvest_date = pd.to_datetime(prod['harvest_date']).date()
        
        crop = prod['crop_type']
        variety = prod['crop_variety']
        market_price = prod['expected_market_price_per_kg']
        actual_price = round(market_price * random.uniform(0.9, 1.1), 2)
        quantity = round(random.uniform(200, 2000), 1)
        revenue = round(quantity * actual_price, 2)
        discount = round(random.uniform(0, 50), 2)
        shipping = round(random.uniform(10, 100), 2)
        net = revenue - discount - shipping
        
        records.append({
            'sales_id': i,
            'production_id': prod['production_id'],
            'crop_type': crop,
            'crop_variety': variety,
            'buyer_name': fake.company(),
            'buyer_type': random.choice(buyer_types),
            'buyer_region': random.choice(regions),
            'channel_type': random.choice(channel_types),
            'transaction_date': fake.date_between(start_date=harvest_date, end_date='today'),
            'quantity_sold_kg': quantity,
            'unit_price_eur': actual_price,
            'total_revenue_eur': revenue,
            'discount_applied_eur': discount,
            'net_revenue_eur': round(net, 2),
            'shipping_cost_eur': shipping,
            'profit_margin_pct': round(random.uniform(5, 25), 2),
            'market_price_per_kg': market_price,
            'price_variance': round(actual_price - market_price, 2)
        })
    return pd.DataFrame(records)


In [29]:
print("Production Sample:")
production_df.head()

Production Sample:


,production_id,farm_name,field_id,field_location,crop_type,crop_variety,planting_date,harvest_date,fertilizer_type,fertilizer_quantity_kg,...,total_yield_kg,avg_yield_per_hectare,soil_ph,rainfall_mm,avg_temperature_c,labor_hours,number_of_workers,machinery_used,expected_market_price_per_kg,expected_total_revenue_eur
0,1,Scott and Sons,F-330,Port Tanyashire,Soybean,Black,2025-04-20,2025-05-05,Compost,128.5,...,8967.9,10917.3,7.30,642.3,17.9,194.2,6,Tractor,1.37,12286.05
1,2,Stewart-Harper,F-718,East Caitlinville,Soybean,Black,2024-11-29,2025-04-19,Compost,133.3,...,1677.9,702.1,6.28,472.0,18.3,197.0,8,Tractor,1.48,2483.33
2,3,Harris PLC,F-891,South Samanthamouth,Soybean,Black,2024-07-25,2024-11-09,NPK 15-15-15,165.2,...,8202.0,5658.0,6.26,341.6,24.9,118.8,8,Plough,1.06,8694.10
3,4,Gill-Wilson,F-366,Port James,Barley,Hulless,2024-09-21,2025-01-28,NPK 15-15-15,64.6,...,1136.0,1564.7,6.01,604.4,16.0,164.9,8,Plough,1.24,1408.63
4,5,Yates Inc,F-493,Lake Jorgefort,Corn,Sweet,2024-07-14,2024-10-17,Urea,267.6,...,4363.2,6401.3,7.29,640.0,14.7,81.9,4,Tractor,1.44,6283.02


In [35]:
# Generate and save data
production_df = generate_production_data()
sales_df = generate_sales_data(production_df)

In [36]:
production_df.to_csv('fact_production.csv', index=False)
sales_df.to_csv('fact_sales.csv', index=False)

In [37]:
# sample output of production data
print("Production Sample:")
production_df.head()


Production Sample:


,production_id,farm_name,field_id,field_location,crop_type,crop_variety,planting_date,harvest_date,fertilizer_type,fertilizer_quantity_kg,...,total_yield_kg,avg_yield_per_hectare,soil_ph,rainfall_mm,avg_temperature_c,labor_hours,number_of_workers,machinery_used,expected_market_price_per_kg,expected_total_revenue_eur
0,1,"Bright, Stone and Lopez",F-979,Lake Joseph,Corn,Dent,2023-10-27,2024-03-27,Urea,184.2,...,3552.8,1459.7,7.31,342.3,17.4,140.6,2,Tractor,1.40,4973.92
1,2,"Mack, Hawkins and Hill",F-492,Turnerfurt,Potato,Russet,2024-07-08,2024-10-30,Urea,275.0,...,3850.6,4602.2,6.84,575.9,12.6,171.4,4,Tractor,0.72,2772.43
2,3,Blake-Hernandez,F-200,Stevenfort,Soybean,Black,2024-07-27,2024-11-30,Urea,251.4,...,9362.7,16033.3,5.61,462.9,14.0,57.1,4,Harvester,0.85,7958.26
3,4,"Moore, Melton and Gonzalez",F-476,Lake Meagantown,Barley,Hulless,2024-01-01,2024-04-11,NPK 15-15-15,76.8,...,4750.6,2127.5,6.76,358.2,13.2,126.4,3,Plough,0.54,2565.34
4,5,Nguyen Group,F-740,Port Stacey,Corn,Dent,2023-06-09,2023-09-15,Compost,252.6,...,2524.5,2643.1,6.47,546.8,21.3,57.4,2,Plough,0.34,858.34


In [38]:
# sample output of sales data
print("\nSales Sample:")
sales_df.head()


Sales Sample:


,sales_id,production_id,crop_type,crop_variety,buyer_name,buyer_type,buyer_region,channel_type,transaction_date,quantity_sold_kg,unit_price_eur,total_revenue_eur,discount_applied_eur,net_revenue_eur,shipping_cost_eur,profit_margin_pct,market_price_per_kg,price_variance
0,1,2874,Soybean,Yellow,Spencer-Underwood,Retailer,Hesse,Direct,2025-04-14,284.6,0.44,125.22,48.48,-8.91,85.65,12.45,0.43,0.01
1,2,5378,Soybean,Black,Mercado PLC,Retailer,Brandenburg,Cooperative,2025-01-03,946.8,1.13,1069.88,47.74,969.82,52.32,6.44,1.14,-0.01
2,3,981,Potato,Russet,Suarez-Davis,Retailer,Brandenburg,Direct,2024-07-14,621.2,0.62,385.14,31.24,339.57,14.33,19.62,0.63,-0.01
3,4,5762,Barley,Hulled,Thompson PLC,Wholesaler,Saxony,Cooperative,2024-11-30,426.1,0.60,255.66,45.47,127.40,82.79,21.64,0.63,-0.03
4,5,8281,Soybean,Black,Lawson-Lawrence,Wholesaler,Bavaria,Cooperative,2025-04-03,1091.5,0.91,993.26,47.46,910.26,35.54,5.40,0.92,-0.01
